# BronzeWork Incremental

Incremental Bronze ingestion with rerun-safe watermark logic.

## Step 1 — Imports and setup

This cell imports the PySpark and Delta helpers used in the notebook,
switches to the correct catalog, and makes sure the Bronze schema
exists before we start loading data.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime
import uuid


In [0]:
spark.sql("use catalog datatocrunch_novacart_adb")

In [0]:
spark.sql("create schema if not exists bronze_schema")

## Step 2 — Bronze control table

This table stores the **watermark** for each source table.

It helps the pipeline remember:

- the latest timestamp already processed
- the latest primary key processed at that timestamp
- how many rows were written in the latest run

This is what makes the Bronze load incremental and rerun-safe.

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS datatocrunch_novacart_adb.bronze_schema.ingestion_control (  
              layer STRING,
              table_name STRING,
              ts_col STRING,
              pk_col STRING,
              last_successful_ts TIMESTAMP,
              last_successful_pk bigint,
              last_run_id string,
              rows_written bigint,
              run_status string,
              updated_at TIMESTAMP
          )
          USING DELTA
          """
)

## Step 3 — Source table configuration

This cell defines which source tables will be loaded into Bronze and which columns should be used as:

- **primary key**
- **timestamp / watermark column**

It also creates a unique `bronze_run_id` for the current pipeline run.

In [0]:
import uuid

tables_config = {
    "orders": {"pk_col": "order_id", "ts_col": "updated_at"},
    "products": {"pk_col": "product_id", "ts_col": "updated_at"},
    "payments": {"pk_col": "payment_id", "ts_col": "processed_at"}
}

bronze_run_id = str(uuid.uuid4())
print("Current Bronze Run ID:", bronze_run_id)

## Step 4 — Helper functions

This cell contains reusable functions:

- `get_last_successful_watermark()` reads the last processed watermark from the control table
- `upsert_bronze_control()` updates the control table after a successful Bronze load

These functions keep the main load logic cleaner and easier to understand.

In [0]:
def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("datacrunch_novacart_adb.bronze_schema.ingestion_control")
        .filter(
            (f.col("layer") == "bronze") &
            (f.col("table_name") == table_name) &
            (f.col("run_status") == "success")
        )
        .orderBy(f.col("updated_at").desc())
        .limit(1)
    )
    
    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

In [0]:
def upsert_bronze_control(table_name,ts_col,pk_col,last_ts,last_pk,rows_written,run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.utcnow()
        )],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark,"datacrunch_novacart_adb.bronze_schema.ingestion_control")
    (dt.alias("t")
        .merge(control_df.alias("s"),"t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col" : "s.ts_col",
            "pk_col" : "s.pk_col",
            "last_successful_ts" : "s.last_successful_ts",
            "last_successful_pk" : "s.last_successful_pk",
            "last_run_id" : "s.last_run_id",
            "rows_written" : "s.rows_written",
            "run_status" : "s.run_status",
            "updated_at" : "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

## Step 5 — Bronze incremental load loop

This is the main Bronze logic.

For each table, the notebook:

1. reads the last watermark
2. reads the source SQL table
3. filters only **new / changed rows**
4. adds Bronze audit columns
5. appends the rows into the Bronze Delta table
6. updates the control table

This is the core incremental loading logic.

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime, timezone

# This safety net prevents the "name 'f' is not defined" error
f = F 

def get_last_successful_watermark(table_name: str):
    ctrl = (
        spark.table("datatocrunch_novacart_adb.bronze_schema.ingestion_control")
        .filter(
            (F.col("layer") == "bronze") &
            (F.col("table_name") == table_name) &
            (F.col("run_status") == "success")
        )
        .orderBy(F.col("updated_at").desc())
        .limit(1)
    )

    rows = ctrl.collect()
    if not rows:
        return None, None

    return rows[0]["last_successful_ts"], rows[0]["last_successful_pk"]

def upsert_bronze_control(table_name, ts_col, pk_col, last_ts, last_pk, rows_written, run_id):
    control_df = spark.createDataFrame(
        [(
            "bronze",
            table_name,
            ts_col,
            pk_col,
            last_ts,
            int(last_pk) if last_pk is not None else None,
            run_id,
            int(rows_written),
            "success",
            datetime.now(timezone.utc)
        )],
        schema="""
            layer string,
            table_name string,
            ts_col string,
            pk_col string,
            last_successful_ts timestamp,
            last_successful_pk bigint,
            last_run_id string,
            rows_written bigint,
            run_status string,
            updated_at timestamp
        """
    )

    dt = DeltaTable.forName(spark, "datatocrunch_novacart_adb.bronze_schema.ingestion_control")
    (dt.alias("t")
        .merge(control_df.alias("s"), "t.table_name = s.table_name and t.layer = s.layer")
        .whenMatchedUpdate(set={
            "ts_col": "s.ts_col",
            "pk_col": "s.pk_col",
            "last_successful_ts": "s.last_successful_ts",
            "last_successful_pk": "s.last_successful_pk",
            "last_run_id": "s.last_run_id",
            "rows_written": "s.rows_written",
            "run_status": "s.run_status",
            "updated_at": "s.updated_at"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

# Core incremental loading logic
for table_name, cfg in tables_config.items():
    pk_col = cfg["pk_col"]
    ts_col = cfg["ts_col"]
    source_table = f"datatocrunch_novacart_sql_connection_catalog.dbo.{table_name}"
    target_table = f"datatocrunch_novacart_adb.bronze_schema.{table_name}_raw"

    last_successful_ts, last_successful_pk = get_last_successful_watermark(table_name)

    ####################### FIX — Truncate watermark to millisecond precision #####
    if last_successful_ts is not None:
        last_successful_ts = last_successful_ts.replace(
            microsecond=(last_successful_ts.microsecond // 1000) * 1000
        )
    ####################### FIX — END #############################################

    print(f"\n=== Processing {table_name} ===")
    print("last_successful_ts =", last_successful_ts)
    print("last_successful_pk =", last_successful_pk)

    ####################### FIX — Truncate source ts to millisecond precision #####
    source_df = spark.read.table(source_table).withColumn(
        ts_col, F.date_trunc("MILLISECOND", F.col(ts_col).cast("timestamp"))
    )
    ####################### FIX — END #############################################

    if last_successful_ts is None:
        rows_to_load = source_df
    else:
        ################### ADDED SECTION - START ##################################
        if last_successful_pk is None:
            rows_to_load = source_df.filter(
                F.col(ts_col) > F.lit(last_successful_ts)
            )
        else:
        ################### ADDED SECTION - END ####################################
            rows_to_load = source_df.filter(
                (F.col(ts_col) > F.lit(last_successful_ts)) |
                (
                    (F.col(ts_col) == F.lit(last_successful_ts)) &
                    (F.col(pk_col).cast("long") > F.lit(int(last_successful_pk)))
                )
            )

    rows_to_load = (
        rows_to_load
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("bronze_run_id", F.lit(bronze_run_id))
        .withColumn("bronze_source_table", F.lit(source_table))
    )

    row_count = rows_to_load.count()

    print(f"{table_name} rows_to_load = {row_count}")

    if row_count == 0:
        print(f"No new rows for {table_name}.")
        upsert_bronze_control(
            table_name,
            ts_col,
            pk_col,
            last_successful_ts,
            last_successful_pk,
            row_count,
            bronze_run_id
        )
        continue

    rows_to_load.write.format("delta").mode("append").saveAsTable(target_table)

    ############################### ADDED SECTION - START #################################
    watermark_row = (
        rows_to_load
        .select(ts_col, pk_col)
        .orderBy(
            F.col(ts_col).desc(),
            F.col(pk_col).cast("long").desc()
        )
        .limit(1)
        .collect()[0]
    )
    ############################### ADDED SECTION - END ###################################

    max_ts = watermark_row[ts_col]
    max_pk = int(watermark_row[pk_col]) if watermark_row[pk_col] is not None else None

    upsert_bronze_control(table_name, ts_col, pk_col, max_ts, max_pk, row_count, bronze_run_id)
    print(f"Wrote {row_count} rows to {target_table}")

## Step 6 — Quick validation

This final cell prints the Bronze row counts and displays the control table so you can verify that the incremental logic is working correctly.

In [0]:
print("Orders Bronze Count : " , spark.sql("select count(*) from datatocrunch_novacart_adb.bronze_schema.orders_raw").collect()[0][0])
print("Products Bronze Count : " , spark.sql("select count(*) from datatocrunch_novacart_adb.bronze_schema.products_raw").collect()[0][0])
print("Payments Bronze Count : " , spark.sql("select count(*) from datatocrunch_novacart_adb.bronze_schema.payments_raw").collect()[0][0])

display(spark.sql("select * from datatocrunch_novacart_adb.bronze_schema.ingestion_control").orderBy("table_name"))